# 01 - Hyperparameter Tuning (Optuna, 4 models)

Notebook ini mencari best hyperparameter untuk 4 model:
- Random Forest
- Extra Trees (via `SKLearnModel` wrapper)
- XGBoost
- LightGBM

Tuning dilakukan pada **target-only** (tanpa covariates) dengan window=120, 3-fold expanding CV. Hasil disimpan ke `saved_models/optuna_tuning_results.joblib` untuk dipakai `phase1_screening.ipynb`.

## Catatan efisiensi
- Fold untuk tuning **3** (bukan 5) biar cepat
- `N_TRIALS = 30` per model - bisa dinaikkan kalau waktu cukup
- Tuning tanpa covariates karena kita cari base hyperparams


## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import joblib
import optuna
from optuna.samplers import TPESampler
from datetime import datetime
import warnings
import gc
import glob

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

df_merged = joblib.load(sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)[0])
print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")
print(f"Columns: {list(df_merged.columns)}")

Data: (2443, 17) | 2015-01-02 to 2025-01-31
Columns: ['date', 'IHSG', 'BI_Rate', 'CPI', 'M2', 'NPL_Ratio', 'GDP', 'USDIDR', 'WTI', 'US_Treasury_10Y', 'Coal', 'Copper', 'Nickel', 'Silver', 'Tin', 'Gold', 'STI']


## 2. Build Target Series (IHSG log-diff, fit once)

In [2]:
# Target only - tuning tidak pakai covariates
target_raw = TimeSeries.from_dataframe(
    df_merged, time_col="date", value_cols="IHSG",
    fill_missing_dates=True, freq="B",
)
target_raw = fill_missing_values(target_raw)

# Transform: log -> diff(1)
target_log = target_raw.map(np.log)
differencer = Diff(lags=1)
target_log_diff = differencer.fit_transform(target_log)

print(f"Target raw:      {len(target_raw)} points")
print(f"Target log-diff: {len(target_log_diff)} points")


Target raw:      2631 points
Target log-diff: 2630 points


## 3. Expanding Window CV Helper

3-fold expanding CV untuk tuning (lebih cepat dari 5-fold). Scaler di-fit per fold pada training data.


In [3]:
WINDOWS         = [20, 120]
HORIZONS        = [1, 5, 20]
N_FOLDS_TUNING  = 3


def expanding_cv_mape(model_builder, series_log_diff, series_raw,
                      window, horizon, n_folds=N_FOLDS_TUNING):
    """3-fold expanding CV untuk satu kombinasi (window, horizon).
    Mengembalikan mean MAPE (%) di level harga IHSG.
    """
    n = len(series_log_diff)
    test_size = int(n * 0.15)
    min_train = int(n * 0.4)
    available = n - min_train - test_size
    step = max(1, available // max(1, n_folds - 1))

    scores = []
    for fold in range(n_folds):
        train_end = min_train + fold * step
        test_end  = min(train_end + test_size, n)
        if test_end > n or train_end >= test_end:
            break

        train  = series_log_diff[:train_end]
        full   = series_log_diff[:test_end]
        test   = series_log_diff[train_end:test_end]

        scaler   = Scaler()
        train_s  = scaler.fit_transform(train)
        full_s   = scaler.transform(full)

        model = model_builder(window, horizon)
        model.fit(train_s)

        forecast_list = model.historical_forecasts(
            series=full_s,
            start=test.start_time(),
            forecast_horizon=horizon,
            stride=horizon,
            retrain=False,
            last_points_only=False,
            verbose=False,
        )
        if isinstance(forecast_list, TimeSeries):
            forecast_list = [forecast_list]

        # Inverse transform ke level harga (sama dengan pipeline notebook 02)
        full_log = series_raw.map(np.log)
        all_dates, all_prices = [], []
        for chunk_scaled in forecast_list:
            chunk_diff = scaler.inverse_transform(chunk_scaled)
            dates = chunk_diff.time_index
            vals  = chunk_diff.values().flatten()
            idx   = series_raw.get_index_at_point(dates[0])
            if idx == 0:
                continue
            anchor     = full_log[idx - 1].values()[0][0]
            log_prices = anchor + np.cumsum(vals)
            all_dates.extend(dates)
            all_prices.extend(np.exp(log_prices))

        if not all_prices:
            scores.append(np.inf)
            continue

        pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
        actual_df = series_raw.to_dataframe().reset_index()
        actual_df.columns = ["date", "actual"]
        eval_df   = pd.merge(actual_df, pred_df, on="date", how="inner")
        y_true    = eval_df["actual"].values
        y_pred    = eval_df["predicted"].values
        mape      = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        scores.append(mape)

        del model
        gc.collect()

    return float(np.mean(scores)) if scores else np.inf


def multi_config_mape(model_builder, series_log_diff, series_raw,
                      windows=WINDOWS, horizons=HORIZONS):
    """Rata-rata MAPE dari semua kombinasi window × horizon.
    Hyperparameter yang dipilih akan optimal untuk semua skenario eksperimen.
    """
    scores = []
    for w in windows:
        for h in horizons:
            s = expanding_cv_mape(model_builder, series_log_diff, series_raw, w, h)
            scores.append(s)
    return float(np.mean(scores))


print(f"CV config: windows={WINDOWS}, horizons={HORIZONS}, n_folds={N_FOLDS_TUNING}")
print(f"Tuning objective: avg MAPE dari {len(WINDOWS) * len(HORIZONS)} kombinasi window×horizon")


CV config: windows=[20, 120], horizons=[1, 5, 20], n_folds=3
Tuning objective: avg MAPE dari 6 kombinasi window×horizon


## 4. Optuna Objectives (4 models)

In [4]:
def rf_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth":         trial.suggest_int("max_depth", 3, 20),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        "max_samples":       trial.suggest_float("max_samples", 0.5, 0.9, step=0.1),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
    }
    def builder(w, h):
        return RandomForestModel(
            lags=w, output_chunk_length=h,
            random_state=42, n_jobs=-1, **params,
        )
    return multi_config_mape(builder, target_log_diff, target_raw)


def et_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth":         trial.suggest_int("max_depth", 3, 20),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
    }
    def builder(w, h):
        return SKLearnModel(
            lags=w, output_chunk_length=h,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params),
        )
    return multi_config_mape(builder, target_log_diff, target_raw)


def xgb_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth":        trial.suggest_int("max_depth", 3, 15),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    def builder(w, h):
        return XGBModel(
            lags=w, output_chunk_length=h,
            random_state=42, n_jobs=-1, **params,
        )
    return multi_config_mape(builder, target_log_diff, target_raw)


def lgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth":         trial.suggest_int("max_depth", 3, 15),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
    }
    def builder(w, h):
        return LightGBMModel(
            lags=w, output_chunk_length=h,
            random_state=42, n_jobs=-1, verbose=-1, **params,
        )
    return multi_config_mape(builder, target_log_diff, target_raw)


OBJECTIVES = {
    "RandomForest": rf_objective,
    "ExtraTrees":   et_objective,
    "XGBoost":      xgb_objective,
    "LightGBM":     lgb_objective,
}
print(f"Objectives defined: {list(OBJECTIVES.keys())}")
print(f"Setiap trial evaluasi {len(WINDOWS) * len(HORIZONS)} kombinasi × {N_FOLDS_TUNING} folds = "
      f"{len(WINDOWS) * len(HORIZONS) * N_FOLDS_TUNING} model fits per trial")


Objectives defined: ['RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM']
Setiap trial evaluasi 6 kombinasi × 3 folds = 18 model fits per trial


## 5. Run Tuning

In [5]:
N_TRIALS = 30  # naikkan kalau waktu cukup

TUNING_RESULTS = {}
total_start = datetime.now()

for model_name, objective_fn in OBJECTIVES.items():
    print(f"\n{'='*60}")
    print(f"Tuning {model_name} ({N_TRIALS} trials)")
    print(f"{'='*60}")

    sampler = TPESampler(seed=42)
    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        study_name=f"IHSG_{model_name}",
    )

    start = datetime.now()
    study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = datetime.now() - start

    TUNING_RESULTS[model_name] = {
        "best_params": study.best_params,
        "best_value":  study.best_value,
    }

    print(f"\n{model_name} done in {elapsed}")
    print(f"  Best MAPE (log-diff space): {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

total_elapsed = datetime.now() - total_start
print(f"\n{'='*60}")
print(f"All tuning done in {total_elapsed}")



Tuning RandomForest (30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]


RandomForest done in 0:14:32.867197
  Best MAPE (log-diff space): 1.2634
  Best params: {'n_estimators': 800, 'max_depth': 3, 'max_features': 0.7, 'max_samples': 0.8, 'min_samples_split': 5, 'min_samples_leaf': 6}

Tuning ExtraTrees (30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]


ExtraTrees done in 0:05:37.044259
  Best MAPE (log-diff space): 1.2625
  Best params: {'n_estimators': 200, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 9, 'min_samples_leaf': 3}

Tuning XGBoost (30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]


XGBoost done in 2:44:54.571324
  Best MAPE (log-diff space): 1.2589
  Best params: {'n_estimators': 600, 'max_depth': 11, 'learning_rate': 0.029464057132418377, 'subsample': 0.8256085472109767, 'colsample_bytree': 0.39195610746628573, 'reg_alpha': 4.723006221405656, 'reg_lambda': 0.0034555486843382047, 'min_child_weight': 8}

Tuning LightGBM (30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]


LightGBM done in 0:29:47.232208
  Best MAPE (log-diff space): 1.2611
  Best params: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.011711509955524094, 'num_leaves': 83, 'subsample': 0.5852620618436457, 'colsample_bytree': 0.3455361150896956, 'reg_alpha': 3.4671276804481113, 'reg_lambda': 4.905556676028774, 'min_child_samples': 42}

All tuning done in 3:34:51.725199


## 6. Save Results

In [6]:
import os
os.makedirs("saved_models", exist_ok=True)

joblib.dump(TUNING_RESULTS, "saved_models/optuna_tuning_results.joblib")
print("Saved: saved_models/optuna_tuning_results.joblib")

# Summary CSV
rows = []
for m, r in TUNING_RESULTS.items():
    row = {"Model": m, "Best_MAPE": round(r["best_value"], 4)}
    row.update(r["best_params"])
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv("saved_models/optuna_tuning_summary.csv", index=False)
display(summary_df)
print("Saved: saved_models/optuna_tuning_summary.csv")


Saved: saved_models/optuna_tuning_results.joblib


,Model,Best_MAPE,n_estimators,max_depth,max_features,max_samples,min_samples_split,min_samples_leaf,learning_rate,subsample,colsample_bytree,reg_alpha,reg_lambda,min_child_weight,num_leaves,min_child_samples
0,RandomForest,1.2634,800,3,0.7,0.8,5.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ExtraTrees,1.2625,200,5,sqrt,NaN,9.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XGBoost,1.2589,600,11,NaN,NaN,NaN,NaN,0.029464,0.825609,0.391956,4.723006,0.003456,8.0,NaN,NaN
3,LightGBM,1.2611,600,10,NaN,NaN,NaN,NaN,0.011712,0.585262,0.345536,3.467128,4.905557,NaN,83.0,42.0


Saved: saved_models/optuna_tuning_summary.csv


## 7. [TAMBAHAN] Run LightGBM Only

Cell ini standalone — tidak butuh menjalankan ulang cell 1-5.
Load hasil RF/ET/XGB yang sudah tersimpan, lalu run LightGBM saja, kemudian gabung dan save ulang.

In [7]:
# import pandas as pd
# import numpy as np
# import joblib
# import optuna
# from optuna.samplers import TPESampler
# from datetime import datetime
# import warnings
# import gc
# import glob

# from darts import TimeSeries
# from darts.models import LightGBMModel
# from darts.dataprocessing.transformers import Scaler, Diff
# from darts.utils.missing_values import fill_missing_values

# warnings.filterwarnings("ignore")
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# # ── Load data & rebuild target series ────────────────────────────────────────
# df_merged = joblib.load(sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)[0])
# print(f"Data loaded: {df_merged.shape}")

# target_raw = TimeSeries.from_dataframe(
#     df_merged, time_col="date", value_cols="IHSG",
#     fill_missing_dates=True, freq="B",
# )
# target_raw = fill_missing_values(target_raw)
# target_log = target_raw.map(np.log)
# differencer = Diff(lags=1)
# target_log_diff = differencer.fit_transform(target_log)
# print(f"Target series: {len(target_log_diff)} points")

# # ── Rebuild CV helper ─────────────────────────────────────────────────────────
# WINDOW = 120
# N_FOLDS_TUNING = 3

# def expanding_cv_mape(model_builder, series_log_diff, series_raw, n_folds=N_FOLDS_TUNING):
#     n = len(series_log_diff)
#     test_size = int(n * 0.15)
#     min_train = int(n * 0.4)
#     available = n - min_train - test_size
#     step = max(1, available // max(1, n_folds - 1))
#     scores = []
#     for fold in range(n_folds):
#         train_end = min_train + fold * step
#         test_end = min(train_end + test_size, n)
#         if test_end > n or train_end >= test_end:
#             break
#         train = series_log_diff[:train_end]
#         full  = series_log_diff[:test_end]
#         test  = series_log_diff[train_end:test_end]
#         scaler = Scaler()
#         train_s = scaler.fit_transform(train)
#         full_s  = scaler.transform(full)
#         model = model_builder()
#         model.fit(train_s)
#         forecasts = model.historical_forecasts(
#             series=full_s, start=test.start_time(),
#             forecast_horizon=1, stride=1,
#             retrain=False, last_points_only=True, verbose=False,
#         )
#         pred_diff  = scaler.inverse_transform(forecasts)
#         pred_dates = pred_diff.time_index
#         pred_vals  = pred_diff.values().flatten()
#         actual_prices, pred_prices = [], []
#         for i, d in enumerate(pred_dates):
#             try:
#                 idx = series_raw.get_index_at_point(d)
#             except Exception:
#                 continue
#             if idx == 0:
#                 continue
#             prev_price = float(series_raw[idx - 1].values()[0][0])
#             curr_price = float(series_raw[idx].values()[0][0])
#             actual_prices.append(curr_price)
#             pred_prices.append(prev_price * np.exp(pred_vals[i]))
#         if not actual_prices:
#             scores.append(np.inf)
#             continue
#         y_true = np.array(actual_prices)
#         y_pred = np.array(pred_prices)
#         scores.append(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
#         del model
#         gc.collect()
#     return float(np.mean(scores))

# # ── LightGBM objective ────────────────────────────────────────────────────────
# def lgb_objective(trial):
#     params = {
#         "n_estimators":      trial.suggest_int("n_estimators", 100, 1000, step=100),
#         "max_depth":         trial.suggest_int("max_depth", 3, 15),
#         "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
#         "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
#         "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.3, 1.0),
#         "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
#         "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
#     }
#     def builder():
#         return LightGBMModel(
#             lags=WINDOW, output_chunk_length=1,
#             random_state=42, n_jobs=-1, verbose=-1, **params,
#         )
#     return expanding_cv_mape(builder, target_log_diff, target_raw)

# # ── Run LightGBM tuning ───────────────────────────────────────────────────────
# N_TRIALS = 30
# print(f"\nTuning LightGBM ({N_TRIALS} trials) ...")

# sampler = TPESampler(seed=42)
# study   = optuna.create_study(direction="minimize", sampler=sampler, study_name="IHSG_LightGBM")
# start   = datetime.now()
# study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
# elapsed = datetime.now() - start

# print(f"\nLightGBM selesai dalam {elapsed}")
# print(f"  Best MAPE : {study.best_value:.4f}")
# print(f"  Best params: {study.best_params}")

# # ── Merge dengan hasil yang sudah ada & save ──────────────────────────────────
# TUNING_RESULTS = joblib.load("saved_models/optuna_tuning_results.joblib")
# print(f"\nHasil sebelumnya: {list(TUNING_RESULTS.keys())}")

# TUNING_RESULTS["LightGBM"] = {"best_params": study.best_params, "best_value": study.best_value}

# joblib.dump(TUNING_RESULTS, "saved_models/optuna_tuning_results.joblib")
# print(f"Updated: {list(TUNING_RESULTS.keys())}")

# rows = []
# for m, r in TUNING_RESULTS.items():
#     row = {"Model": m, "Best_MAPE": round(r["best_value"], 4)}
#     row.update(r["best_params"])
#     rows.append(row)
# pd.DataFrame(rows).to_csv("saved_models/optuna_tuning_summary.csv", index=False)
# print("Saved: optuna_tuning_results.joblib & optuna_tuning_summary.csv")